# Can we classify and route many short camping-store questions without paying frontier-model latency for every request?

## 1. Before You Begin

Contoso Outdoors receives short questions that can go to sales, product support,
returns, or safety review. This notebook focuses on one capability:
**low-latency request classification**.

Deploy `gpt-6-luna` in Microsoft Foundry, configure section 2, and review the
[repository quickstart](../../quickstart/README.md). Use the
[Azure OpenAI pricing page](https://azure.microsoft.com/pricing/details/azure-openai/)
for current rates.


In [ ]:
# 2. Verify your environment
import os
from pathlib import Path

from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()


def find_assets() -> Path:
    """Find the shared data whether Jupyter starts here or at the repo root."""
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / "models/azure-openai/shared/contoso-outdoors"
        if (candidate / "products.json").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not find models/azure-openai/shared/contoso-outdoors. "
        "Run this notebook from a checkout of the model-releases repository."
    )


required = ["AZURE_OPENAI_ENDPOINT", "AZURE_OPENAI_API_KEY", "AZURE_OPENAI_GPT_6_LUNA_DEPLOYMENT"]
missing = [name for name in required if not os.getenv(name)]
if missing:
    raise EnvironmentError(
        f"Missing {missing}. Copy scripts/sample.env to .env, add the values, "
        "and review models/quickstart/README.md."
    )

endpoint = os.environ["AZURE_OPENAI_ENDPOINT"].rstrip("/")
base_url = endpoint if endpoint.endswith("/openai/v1") else f"{endpoint}/openai/v1"
client = OpenAI(api_key=os.environ["AZURE_OPENAI_API_KEY"], base_url=f"{base_url}/")
deployment = os.environ["AZURE_OPENAI_GPT_6_LUNA_DEPLOYMENT"]
assets = find_assets()

expected_assets = [
    assets / "products.json",
    assets / "manuals/product_info_1.md",
    assets / "manuals/product_info_2.md",
    assets / "images/product_1.webp",
    assets / "images/product_2.webp",
]
missing_assets = [str(path) for path in expected_assets if not path.is_file()]
if missing_assets:
    raise FileNotFoundError(f"Shared Contoso Outdoors files are missing: {missing_assets}")

print(f"Environment ready for deployment: {deployment}")
print(f"Shared assets: {assets}")


## 3. Fix the routing task

We use four deterministic questions tied to the same tent and backpack subset.
The images make the shared retail scenario visible, but the classifier receives
only the question and the allowed route definitions.


In [ ]:
# 4. Load the local product evidence
import base64
import json

from IPython.display import HTML, display

# These are the only product records used in this phase.
products = json.loads((assets / "products.json").read_text(encoding="utf-8"))
manuals = {
    product["id"]: (assets / product["manual"]).read_text(encoding="utf-8")
    for product in products
}

def display_webp(path: Path, width: int = 240) -> None:
    """Render a local WebP through HTML because IPython Image cannot embed it."""
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    display(HTML(f'<img src="data:image/webp;base64,{encoded}" width="{width}">'))


for product in products:
    print(f'{product["id"]}: {product["name"]} (${product["price"]})')
    display_webp(assets / product["images"][0])

assert [product["id"] for product in products] == [1, 2]


questions = [
    "Is the TrailMaster X4 suitable for a four-person winter expedition?",
    "How should I clean the Adventurer Pro Backpack?",
    "Can I return the backpack after 45 days with Gold membership?",
    "Which of these products costs less?",
]
routes = {
    "sales": "price or product-choice questions",
    "product_support": "setup, care, or ordinary usage questions",
    "returns": "return-policy or warranty questions",
    "safety_review": "requests involving unsupported or potentially unsafe use",
}


## 5. Route each question

We disable reasoning for this short classification path, require a strict JSON
schema, and time each call. These timings describe this environment and request
set only; they are not a model benchmark.


In [ ]:
# 6. Classify the fixed request set
import time

schema = {
    "type": "object",
    "properties": {
        "route": {"type": "string", "enum": list(routes)},
        "reason": {"type": "string"},
    },
    "required": ["route", "reason"],
    "additionalProperties": False,
}
results = []

for question in questions:
    prompt = (
        "Route the shopper question using only these route definitions:\n"
        f"{json.dumps(routes, indent=2)}\n\nQuestion: {question}"
    )
    started = time.perf_counter()
    response = client.responses.create(
        model=deployment,
        input=prompt,
        reasoning={"effort": "none"},
        text={
            "format": {
                "type": "json_schema",
                "name": "route_decision",
                "strict": True,
                "schema": schema,
            }
        },
    )
    elapsed_ms = (time.perf_counter() - started) * 1000
    decision = json.loads(response.output_text)
    assert decision["route"] in routes
    results.append(
        {
            "question": question,
            "route": decision["route"],
            "reason": decision["reason"],
            "elapsed_ms": round(elapsed_ms, 1),
        }
    )

for result in results:
    print(json.dumps(result, indent=2))
print("\nThese elapsed times are observations from this run, not a benchmark.")


## 7. Your Turn to Explore

- Add an ambiguous question and define an `escalation` route before running it.
- Repeat the same fixed set and compare route stability.
- Shorten the route descriptions and inspect which distinctions are lost.


## 8. Summary

We used GPT-6 Luna for one capability: classifying short requests into a
closed route set. Strict output made parsing deterministic, while local timing
made latency visible without overgeneralizing. The
[chat-completion primer](../../../docs/primers/chat-completion.md) explains
when a direct request is preferable to a deeper reasoning workflow.


## 9. References

- [GPT-6 Luna model card](https://ai.azure.com/catalog/models/gpt-6-luna) — model positioning.
- [GPT-6 Astra, Sol, and Luna in Microsoft Foundry](https://azure.microsoft.com/en-us/blog/gpt-6-astra-sol-and-luna-for-production-agents-in-microsoft-foundry/) — family release and deployment guidance.
- [Use the Azure OpenAI Responses API](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/responses) — Python, structured output, and usage patterns.
- [Azure OpenAI reasoning models](https://learn.microsoft.com/en-us/azure/foundry/openai/how-to/reasoning) — supported reasoning effort controls.
- [Foundry Models sold by Azure](https://learn.microsoft.com/en-us/azure/foundry/foundry-models/concepts/models-sold-directly-by-azure#gpt-56) — verified model version.
